# S7 — Ejercicio RESUELTO: Detector de buses y camiones

**Caso de estudio**: una empresa de logistica quiere monitorizar automáticamente
el trafico de vehiculos pesados en sus centros de distribución. Tu tarea es construir
un detector que identifique y localice **buses** y **camiones** en imágenes reales.

Implementaras dos detectores completos paso a paso:

| Paso | Tarea |
|------|-------|
| 1 | Cargar y explorar el dataset |
| 2 | Preparar los datos para Faster R-CNN |
| 3 | Crear, entrenar y evaluar un Faster R-CNN |
| 4 | Preparar los datos para YOLOv8 |
| 5 | Entrenar y evaluar un YOLOv8 |
| 6 | Comparar ambos modelos |
| 7 | Preguntas de reflexion |

> Todas las herramientas necesarias estan en el notebook de clase `S7_CV_object_detection.ipynb`.

In [ ]:
# OPCIONAL — Solo si usas Google Colab y quieres guardar el dataset en Drive
# (evita re-descargar cada sesion)
# Descomenta y ejecuta esta celda ANTES de la celda de descarga.

# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.chdir('/content/drive/MyDrive/CV1')  # cambia esta ruta a tu carpeta en Drive
# print('Directorio de trabajo:', os.getcwd())

## Configuración

In [ ]:
# Instalar paquetes y descargar dataset (ejecutar una vez)
import subprocess, sys, urllib.request, tarfile, os
from pathlib import Path

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'torch', 'torchvision', 'ultralytics', 'Pillow', 'tqdm', 'scikit-learn'])

DATASET_DIR = Path('open-images-bus-trucks')
if not DATASET_DIR.exists():
    url = 'https://www.dropbox.com/s/agmzwk95v96ihic/open-images-bus-trucks.tar.xz?dl=1'
    dest = 'open-images-bus-trucks.tar.xz'
    print('Descargando dataset (~500 MB)... (puede tardar 2-5 min)')
    urllib.request.urlretrieve(url, dest)
    print('Descarga completa. Extrayendo...')
    with tarfile.open(dest, 'r:xz') as tar:
        tar.extractall('.')
    os.remove(dest)
    print('Dataset listo.')
else:
    print('Dataset ya existe.')

IMAGE_DIR = DATASET_DIR / 'images'
CSV_PATH  = DATASET_DIR / 'df.csv'
print('Imagenes: ' + str(len(list(IMAGE_DIR.glob('*.jpg')))))

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import torchvision
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} — Usando: {device}')

In [ ]:
# ── MODO DE EJECUCIÓN ───────────────────────────────────────────
# False → usa modelos COCO preentrenados directamente (bus y truck ya están
#          en COCO, no hace falta fine-tuning para ver el pipeline completo).
# True  → ejecuta el entrenamiento completo (puede tardar mucho sin GPU).
ENTRENAR = False
# ─────────────────────────────────────────────────────────────────────────
if not ENTRENAR:
    print('ENTRENAR = False — se usarán modelos COCO preentrenados.')
    print('Cambia a True para ejecutar el fine-tuning completo.')

## Paso 1 — Cargar y explorar el dataset

Carga el archivo CSV de anotaciones y responde:
- ¿Cuantas imágenes tiene el dataset?
- ¿Cuantas anotaciones (filas) hay en total?
- ¿Cuantos buses y cuantos camiones?
- ¿Que columnas tiene el CSV?

Muestra las primeras 5 filas.

In [ ]:
# PASO 1a: Carga el CSV y explora su contenido
df = pd.read_csv(CSV_PATH)

print(f'Total anotaciones (filas): {len(df)}')
print(f'Imagenes unicas: {df.ImageID.nunique()}')
print(f'\nDistribucion de clases:')
print(df.LabelName.value_counts())
print(f'\nColumnas: {list(df.columns)}')
df.head()

Ahora visualiza **4 imágenes** del dataset con sus bounding boxes anotados.

> **Pista**: las coordenadas `XMin, YMin, XMax, YMax` estan normalizadas (0-1).
> Multiplica por el ancho/alto de la imagen para obtener píxeles.
> Usa `mpatches.Rectangle` para dibujar las cajas.

In [ ]:
# PASO 1b: Visualiza 4 imágenes con sus bounding boxes
sample_ids = df.ImageID.unique()[:4]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
colors = {'Bus': 'lime', 'Truck': 'red'}

for ax, img_id in zip(axes, sample_ids):
    img_path = IMAGE_DIR / f'{img_id}.jpg'
    img = Image.open(img_path)
    w, h = img.size
    ax.imshow(img)

    data = df[df.ImageID == img_id]
    for _, row in data.iterrows():
        x1 = row.XMin * w
        y1 = row.YMin * h
        x2 = row.XMax * w
        y2 = row.YMax * h
        color = colors.get(row.LabelName, 'cyan')
        rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                   linewidth=2, edgecolor=color,
                                   facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1 - 5, row.LabelName, color=color, fontsize=8,
                fontweight='bold', backgroundcolor='black')

    ax.set_title(img_id[:10], fontsize=9)
    ax.set_axis_off()

plt.tight_layout()
plt.show()

## Paso 2 — Preparar los datos para Faster R-CNN

Faster R-CNN (torchvision) espera:
- **Imagen**: tensor `(C, H, W)` float en [0, 1]
- **Target**: diccionario con `boxes` (float tensor, píxeles) y `labels` (long tensor)
- **Clase 0 = fondo** (reservada), asi que bus=1, truck=2

### 2a — Define el mapeo de clases y haz el split train/val

In [ ]:
from sklearn.model_selection import train_test_split

# PASO 2a: Define el mapeo de clases y haz el split
label2idx = {'Bus': 1, 'Truck': 2}
idx2label = {0: 'background', 1: 'Bus', 2: 'Truck'}
NUM_CLASSES = 3  # fondo + bus + truck

image_ids = df.ImageID.unique()
train_ids, val_ids = train_test_split(image_ids, test_size=0.2, random_state=42)

print(f'Clases: {label2idx}')
print(f'Train: {len(train_ids)} | Val: {len(val_ids)}')

In [ ]:
# RECORTE PARA CLASE — entrena en minutos en vez de horas
# Con el dataset completo (~13.000 train) Faster R-CNN tarda horas en CPU.
# Usamos un subconjunto para que sea viable en clase.
# En produccion o con GPU potente, comenta estas dos lineas.
train_ids = train_ids[:800]
val_ids   = val_ids[:200]
print(f'Train: {len(train_ids)} imagenes | Val: {len(val_ids)} imagenes')
print('Recomendacion: ejecuta este notebook en Google Colab con GPU T4.')

### 2b — Implementa la clase `BusTruckDataset`

El metodo `__getitem__` debe:
1. Cargar la imagen con PIL y redimensionar a `self.img_size x self.img_size`
2. Convertir a tensor float [0, 1] con forma `(C, H, W)`
3. Leer las anotaciones del DataFrame para esa imagen
4. Convertir las coordenadas normalizadas a píxeles
5. Devolver `(img_tensor, target_dict)`

> **Pista**: para una busqueda rápida de imágenes, crea un diccionario
> `{nombre_sin_extension: ruta_completa}` en `__init__`.

In [ ]:
class BusTruckDataset(Dataset):

    def __init__(self, image_ids, df, image_dir, img_size=480):
        self.image_ids = image_ids
        self.df = df
        self.image_dir = image_dir
        self.img_size = img_size
        self.paths = {p.stem: p for p in image_dir.glob('*.jpg')}

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]

        # 1. Cargar imagen, redimensionar, convertir a tensor (C, H, W) float [0,1]
        img = Image.open(self.paths[image_id]).resize(
            (self.img_size, self.img_size)).convert('RGB')
        img = torch.tensor(np.array(img), dtype=torch.float32).permute(2, 0, 1) / 255.0

        # 2. Leer anotaciones y construir target dict
        rows = self.df[self.df.ImageID == image_id]
        boxes = []
        labels = []
        for _, row in rows.iterrows():
            x1 = row.XMin * self.img_size
            y1 = row.YMin * self.img_size
            x2 = row.XMax * self.img_size
            y2 = row.YMax * self.img_size
            boxes.append([x1, y1, x2, y2])
            labels.append(label2idx[row.LabelName])

        target = {
            'boxes':  torch.tensor(boxes, dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.long),
        }
        return img, target

def collate_fn(batch):
    return tuple(zip(*batch))

In [ ]:
if ENTRENAR:
    # Crear datasets y dataloaders
    train_ds = BusTruckDataset(train_ids, df, IMAGE_DIR)
    val_ds   = BusTruckDataset(val_ids, df, IMAGE_DIR)

    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,
                              collate_fn=collate_fn, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False,
                              collate_fn=collate_fn, drop_last=True)

    # Verificar: carga un batch y muestra shapes
    imgs, targets = next(iter(train_loader))
    print(f'Batch: {len(imgs)} imagenes de shape {imgs[0].shape}')
    print(f'Target[0] boxes: {targets[0]["boxes"].shape}, labels: {targets[0]["labels"]}')
else:
    print('DataLoaders omitidos (ENTRENAR = False).')
    print('En modo demo se predice directamente sobre imagenes del dataset.')

## Paso 3 — Faster R-CNN: crear, entrenar y evaluar

### 3a — Crear el modelo

Carga `fasterrcnn_resnet50_fpn` preentrenado y reemplaza la cabeza de clasificación
para nuestras `NUM_CLASSES` clases.

> **Pista**: la cabeza se reemplaza con `FastRCNNPredictor(in_features, num_classes)`.
> El número de `in_features` se obtiene de `model.roi_heads.box_predictor.cls_score.in_features`.

In [ ]:
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

if ENTRENAR:
    model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
else:
    # Modo demo: COCO preentrenado (bus=6, truck=8)
    model = fasterrcnn_resnet50_fpn(
        weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    )
    COCO_TO_LOCAL = {6: 1, 8: 2}
    idx2label = {0: 'background', 1: 'Bus', 2: 'Truck'}
    print('Modelo COCO preentrenado (sin fine-tuning).')

model = model.to(device)
print(f'Parametros: {sum(p.numel() for p in model.parameters()):,}')

### 3b — Entrenar el modelo

Implementa el bucle de entrenamiento. Recuerda:
- Faster R-CNN devuelve un **diccionario de losses** cuando esta en `.train()` mode
- El loss total es la **suma** de los 4 componentes
- Para validacion: usa `torch.no_grad()` pero mantiene `.train()` (para obtener losses)

Entrena **3 épocas** (suficiente para una demo; en producción usarias 10-20).

In [ ]:
if ENTRENAR:
    # PASO 3b: Configura optimizer y entrena
    optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
    NUM_EPOCHS = 3
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(NUM_EPOCHS):
        model.train()
        train_losses = []
        for images, targets in tqdm(train_loader, desc=f'Epoca {epoch+1} [train]'):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            total_loss = sum(loss_dict.values())
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
            train_losses.append(total_loss.item())

        val_losses = []
        with torch.no_grad():
            model.train()
            for images, targets in tqdm(val_loader, desc=f'Epoca {epoch+1} [val]  '):
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
                loss_dict = model(images, targets)
                total_loss = sum(loss_dict.values())
                val_losses.append(total_loss.item())

        history['train_loss'].append(np.mean(train_losses))
        history['val_loss'].append(np.mean(val_losses))
        print(f'  train={history["train_loss"][-1]:.4f}  val={history["val_loss"][-1]:.4f}')

    print('Entrenamiento completado.')
else:
    print('Entrenamiento omitido (ENTRENAR = False).')

### 3c — Graficar las curvas de loss

Dibuja un gráfico con el loss de entrenamiento y validacion por época.

In [ ]:
if ENTRENAR:
    fig, ax = plt.subplots(figsize=(8, 4))
    epochs = range(1, NUM_EPOCHS + 1)
    ax.plot(epochs, history['train_loss'], 'o-', label='Train loss')
    ax.plot(epochs, history['val_loss'], 's--', label='Val loss')
    ax.set_xlabel('Epoca'); ax.set_ylabel('Loss')
    ax.set_title('Faster R-CNN — Curvas de loss')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('Curvas de loss omitidas (ENTRENAR = False).')

### 3d — Predecir y visualizar

Implementa la función `predecir` que:
1. Pone el modelo en `.eval()`
2. Ejecuta `model([img_tensor])` con `torch.no_grad()`
3. Filtra las predicciones por `conf_threshold`
4. Aplica NMS con `torchvision.ops.nms`
5. Devuelve `(boxes, labels, scores)` como arrays numpy

Luego visualiza predicciones sobre **8 imágenes de validacion**.

In [ ]:
from torchvision.ops import nms

def predecir(model, img_tensor, conf_threshold=0.5, nms_threshold=0.3):
    model.eval()
    with torch.no_grad():
        preds = model([img_tensor.to(device)])[0]

    boxes  = preds['boxes'].cpu()
    labels = preds['labels'].cpu()
    scores = preds['scores'].cpu()

    # Con modelo COCO: filtrar solo bus (6) y truck (8)
    if not ENTRENAR:
        mask_cls = (labels == 6) | (labels == 8)
        boxes, labels, scores = boxes[mask_cls], labels[mask_cls], scores[mask_cls]

    mask = scores >= conf_threshold
    boxes, labels, scores = boxes[mask], labels[mask], scores[mask]

    if len(boxes) > 0:
        keep = nms(boxes, scores, nms_threshold)
        boxes, labels, scores = boxes[keep], labels[keep], scores[keep]

    # Remap COCO → nuestros indices
    if not ENTRENAR and len(labels) > 0:
        labels_np = labels.numpy().copy()
        for coco_id, local_id in COCO_TO_LOCAL.items():
            labels_np[labels.numpy() == coco_id] = local_id
        return boxes.cpu().numpy(), labels_np, scores.cpu().numpy()

    return boxes.cpu().numpy(), labels.cpu().numpy(), scores.cpu().numpy()

# Visualiza 8 imagenes con predicciones
# Con fine-tuning: val_ds; sin fine-tuning: imagenes del dataset directamente
sample_images = sorted(IMAGE_DIR.glob('*.jpg'))[:8]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax in axes.flat:
    ax.set_axis_off()

colors_pred = {1: 'lime', 2: 'red'}

for i, ax in enumerate(axes.flat):
    if i >= len(sample_images):
        break
    img_pil = Image.open(sample_images[i]).resize((480, 480))
    img_t = torch.tensor(
        np.array(img_pil.convert('RGB')), dtype=torch.float32
    ).permute(2, 0, 1) / 255.0
    boxes, labels, scores = predecir(model, img_t)

    ax.imshow(img_pil)
    for box, label, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = box
        color = colors_pred.get(int(label), 'cyan')
        rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-5, f'{idx2label.get(int(label), label)} {score:.2f}',
                color=color, fontsize=8, fontweight='bold',
                backgroundcolor='black')

titulo = 'Faster R-CNN (fine-tuned)' if ENTRENAR else 'Faster R-CNN COCO preentrenado'
plt.suptitle(titulo + ' — Predicciones', fontsize=14)
plt.tight_layout()
plt.show()

## Paso 4 — Preparar los datos para YOLOv8

Ultralytics espera una estructura de carpetas específica:
```
yolo-bus-trucks/
  images/train/  images/val/
  labels/train/  labels/val/
```

Cada `.txt` de labels tiene una línea por objeto: `class_id cx cy w h` (normalizado 0-1).

El dataset ya incluye labels en formato YOLO en `open-images-bus-trucks/yolo_labels/all/`.
Reorganiza los archivos en la estructura correcta.

In [ ]:
import shutil

YOLO_DIR = Path('yolo-bus-trucks')
YOLO_LABELS_DIR = DATASET_DIR / 'yolo_labels' / 'all'

if ENTRENAR:
    for split, ids in [('train', train_ids), ('val', val_ids)]:
        (YOLO_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
        (YOLO_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)
        for img_id in tqdm(ids, desc=f'Copiando {split}'):
            src_img = IMAGE_DIR / f'{img_id}.jpg'
            dst_img = YOLO_DIR / 'images' / split / f'{img_id}.jpg'
            if src_img.exists() and not dst_img.exists():
                shutil.copy2(src_img, dst_img)
            src_lbl = YOLO_LABELS_DIR / 'labels' / f'{img_id}.txt'
            dst_lbl = YOLO_DIR / 'labels' / split / f'{img_id}.txt'
            if src_lbl.exists() and not dst_lbl.exists():
                shutil.copy2(src_lbl, dst_lbl)
    for split in ['train', 'val']:
        n = len(list((YOLO_DIR / 'images' / split).glob('*.jpg')))
        print(f'{split}: {n} imagenes')
else:
    print('Preparacion de datos YOLO omitida (ENTRENAR = False).')

In [ ]:
if ENTRENAR:
    yaml_path = Path('bus_truck.yaml')
    yaml_content = (
        f'path: {YOLO_DIR.resolve()}\n'
        'train: images/train\n'
        'val: images/val\n'
        '\nnames:\n'
        '  0: Bus\n'
        '  1: Truck\n'
    )
    yaml_path.write_text(yaml_content)
    print(f'Archivo YAML creado: {yaml_path}')
    print(yaml_content)
else:
    print('YAML omitido (ENTRENAR = False).')

## Paso 5 — Entrenar y evaluar YOLOv8

Con Ultralytics el entrenamiento se reduce a tres líneas:

```python
from ultralytics import YOLO
model = YOLO('yolov8m.pt')
model.train(data='config.yaml', epochs=N)
```

Entrena **3 épocas** y después evalua con `model.val()` para obtener mAP.

In [ ]:
from ultralytics import YOLO

if ENTRENAR:
    yolo_model = YOLO('yolov8m.pt')
    yolo_model.train(data=str(yaml_path), epochs=3, imgsz=480, batch=8, verbose=True)
    print('Entrenamiento completado.')
else:
    yolo_model = YOLO('yolov8m.pt')
    print('YOLOv8m COCO preentrenado cargado (sin fine-tuning).')

In [ ]:
if ENTRENAR:
    metrics = yolo_model.val()
    print(f'mAP50:    {metrics.box.map50:.4f}')
    print(f'mAP50-95: {metrics.box.map:.4f}')
else:
    print('Validacion omitida (ENTRENAR = False).')
    print('mAP50 esperado con fine-tuning (800 imgs, 3 ep): ~0.40–0.60')

### Visualizar predicciones de YOLOv8

Predice sobre **8 imágenes de validacion** y dibuja los resultados.

> **Pista**: `result.boxes.xyxy` da las coordenadas, `result.boxes.cls` las clases,
> `result.boxes.conf` las confianzas, y `result.names` el mapeo id→nombre.

In [ ]:
import random

if ENTRENAR:
    yolo_img_source = sorted((YOLO_DIR / 'images' / 'val').glob('*.jpg'))
else:
    yolo_img_source = sorted(IMAGE_DIR.glob('*.jpg'))

sample = random.sample(yolo_img_source, min(8, len(yolo_img_source)))

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax in axes.flat:
    ax.set_axis_off()

for ax, img_path in zip(axes.flat, sample):
    result = yolo_model(str(img_path), verbose=False)[0]
    ax.imshow(Image.open(img_path))
    for box, cls_id, conf in zip(
        result.boxes.xyxy.cpu().numpy(),
        result.boxes.cls.cpu().numpy(),
        result.boxes.conf.cpu().numpy(),
    ):
        name = result.names[int(cls_id)]
        if name not in ('bus', 'truck', 'Bus', 'Truck'):
            continue
        if conf < 0.5:
            continue
        x1, y1, x2, y2 = box
        color = 'lime' if name.lower() == 'bus' else 'red'
        rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-5, f'{name} {conf:.2f}',
                color=color, fontsize=8, fontweight='bold',
                backgroundcolor='black')

titulo = 'YOLOv8 (fine-tuned)' if ENTRENAR else 'YOLOv8 COCO preentrenado'
plt.suptitle(titulo, fontsize=14)
plt.tight_layout()
plt.show()

## Paso 6 — Comparar ambos modelos

Elige **4 imágenes** del set de validacion y muestra las predicciones de
ambos modelos lado a lado (2 filas: Faster R-CNN arriba, YOLOv8 abajo).

Analiza visualmente: ¿cual localiza mejor? ¿cual comete menos errores?

In [ ]:
import random

compare_images = random.sample(sorted(IMAGE_DIR.glob('*.jpg')), 4)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for col, img_path in enumerate(compare_images):
    img_pil = Image.open(img_path).resize((480, 480))
    img_t = torch.tensor(
        np.array(img_pil.convert('RGB')), dtype=torch.float32
    ).permute(2, 0, 1) / 255.0

    # ── Fila 1: Faster R-CNN ───────────────────────────────────────
    ax = axes[0, col]
    ax.imshow(img_pil)
    boxes, labels, scores = predecir(model, img_t)
    for box, label, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = box
        color = colors_pred.get(int(label), 'cyan')
        rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-5, f'{idx2label.get(int(label), label)} {score:.2f}',
                color=color, fontsize=7, backgroundcolor='black')
    ax.set_axis_off()

    # ── Fila 2: YOLOv8 ──────────────────────────────────────────
    ax2 = axes[1, col]
    ax2.imshow(img_pil)
    result = yolo_model(str(img_path), verbose=False)[0]
    for box, cls_id, conf in zip(
        result.boxes.xyxy.cpu().numpy(),
        result.boxes.cls.cpu().numpy(),
        result.boxes.conf.cpu().numpy(),
    ):
        name = result.names[int(cls_id)]
        if name not in ('bus', 'truck', 'Bus', 'Truck'):
            continue
        if conf < 0.5:
            continue
        x1, y1, x2, y2 = box
        color = 'lime' if name.lower() == 'bus' else 'red'
        rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor=color, facecolor='none')
        ax2.add_patch(rect)
        ax2.text(x1, y1-5, f'{name} {conf:.2f}',
                 color=color, fontsize=7, backgroundcolor='black')
    ax2.set_axis_off()

axes[0, 0].set_title('Faster R-CNN', fontsize=11, fontweight='bold', loc='left')
axes[1, 0].set_title('YOLOv8', fontsize=11, fontweight='bold', loc='left')

plt.suptitle('Comparacion: Faster R-CNN vs YOLOv8', fontsize=14)
plt.tight_layout()
plt.show()

## Paso 7 — Preguntas de reflexion

1. **Formato de datos**: Faster R-CNN usa coordenadas en píxeles `[x1, y1, x2, y2]`
   y YOLO usa coordenadas normalizadas centradas `[cx, cy, w, h]`.
   ¿Que ventaja tiene cada formato?

2. **Velocidad de entrenamiento**: ¿cual de los dos modelos fue más rápido
   de entrenar? ¿A que crees que se debe la diferencia?

3. **Facilidad de uso**: compara el codigo necesario para entrenar cada modelo.
   ¿Que gestiona Ultralytics automáticamente que en Faster R-CNN hay que hacer manualmente?

4. **Confianza**: observa las predicciones con confianza más baja de cada modelo.
   ¿Corresponden a errores o a objetos parcialmente ocultos?

5. **Aplicación real**: para el caso de uso planteado (monitorizar trafico de
   vehiculos pesados en centros de distribución), ¿que modelo recomendarias
   y con que configuración? Justifica tu respuesta.

### Respuestas modelo

**1. Formato de datos**

- **Píxeles `[x1, y1, x2, y2]`** (Faster R-CNN): fácil de visualizar directamente
  sobre la imagen, intuitivo para calcular áreas (`(x2-x1)*(y2-y1)`), pero depende
  de la resolución de la imagen. Si redimensionas la imagen, hay que recalcular las coordenadas.

- **Normalizado centrado `[cx, cy, w, h]`** (YOLO): independiente de la resolución
  (valores entre 0 y 1), las coordenadas funcionan para cualquier tamaño de imagen sin
  conversion. Centro + ancho/alto es más natural para pensar en objetos como "región centrada
  en un punto". Más eficiente para la regresión de la red.

**2. Velocidad de entrenamiento**

YOLOv8 suele ser más rápido por varias razones:
- **Arquitectura single-stage**: no necesita la fase RPN + RoI Pooling de Faster R-CNN
- **Optimizaciones de Ultralytics**: data augmentation en GPU, mixed precisión automático,
  learning rate scheduling optimizado
- **Anchor-free**: no necesita generar y filtrar miles de anchors en cada imagen

Faster R-CNN con 2 stages hace más operaciones por imagen, lo que lo hace más lento por época.

**3. Facilidad de uso**

Ultralytics gestiona automáticamente:
- Data augmentation (mosaic, mixup, HSV jitter, flip)
- Learning rate scheduling (cosine annealing con warmup)
- Mixed precisión training (FP16)
- Checkpointing (guarda el mejor modelo automáticamente)
- Logging de métricas (mAP, loss por componente)
- Estructura de directorios para datos (solo necesitas un YAML)
- EMA (Exponential Moving Average) del modelo

En Faster R-CNN hay que implementar manualmente: el Dataset, el DataLoader con collate_fn,
el bucle de entrenamiento, el scheduler, las métricas de evaluación, el guardado de checkpoints, etc.

**4. Confianza**

Las predicciones de baja confianza suelen corresponder a:
- Objetos parcialmente **ocluidos** (un camion detras de otro)
- Objetos en los **bordes** de la imagen (cortados)
- Objetos **lejanos** (pequenos en la imagen, pocos píxeles)
- Objetos **ambiguos** (un vehiculo que podria ser bus o camion)

Raramente son errores completos en un modelo bien entrenado — más bien reflejan
incertidumbre legitima del modelo. El umbral de confianza es un tradeoff:
alto = menos falsos positivos, bajo = menos falsos negativos.

**5. Aplicación real**

Para monitorizar trafico en centros de distribución, recomendaria **YOLOv8** porque:
- **Velocidad**: necesitamos procesamiento en tiempo real o cerca de real-time
  para monitorizar trafico continuo. YOLOv8 puede correr a 30+ FPS.
- **Facilidad de despliegue**: Ultralytics permite exportar a ONNX, TensorRT, CoreML,
  lo que facilita el despliegue en hardware edge (cámaras IP, Jetson Nano).
- **Precisión suficiente**: para 2 clases (bus/truck) con imágenes desde una posición
  fija, YOLOv8 ofrece precisión comparable a Faster R-CNN.

Configuración recomendada:
- Modelo: `yolov8m` (balance velocidad/precisión)
- Resolución: 640px (suficiente para vehiculos grandes)
- Confianza: 0.5 (evitar falsos positivos en un entorno controlado)
- Agregar **tracking** (ByteTrack via Ultralytics) para contar vehiculos únicos
  y no contar el mismo vehiculo múltiples veces